# 1: Imports and Path Setup


In [1]:
import os
import sys
import numpy as np
import tensorflow as tf

In [2]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'
FIGURES_DIR  = os.path.join(PROJECT_ROOT, 'figures')


In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Transformer_Mine

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Transformer_Mine


In [4]:
from attention import scaled_dot_product_attention
from data_pipeline import (
    create_padding_mask,
    create_look_ahead_mask,
    create_decoder_mask,
)

#2: Mathematical Foundation (Documentation)

From the paper (Section 3.2.2):

$$
\mathrm{MultiHead}(Q, K, V)
=
\mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h)W_O
$$

where:

$$
\mathrm{head}_i
=
\mathrm{Attention}
(QW_{Q_i}, KW_{K_i}, VW_{V_i})
$$

Each head learns a **DIFFERENT linear projection** of $Q$, $K$, $V$, allowing the model to jointly attend to information from different representation subspaces at different positions.

---

## Learned Projection Matrices (Per Head)

- $W_{Q_i}$ : `(d_model, d_k)` where $d_k = d_model / num_heads$
- $W_{K_i}$ : `(d_model, d_k)`
- $W_{V_i}$ : `(d_model, d_v)` where $d_v = d_model / num_heads$
- $W_O$ : `(h * d_v, d_model) = (d_model, d_model)`

---

## Implementation Note

Rather than keeping $h$ separate $W_Q$ matrices of shape `(d_model, d_k)`, we use **ONE Dense layer** of shape `(d_model, d_model)` and then **SPLIT** the output into $h$ heads.

This is mathematically equivalent and more efficient.

---

## Step-by-Step

### 1. Linear Projections

$$
Q_{\text{proj}} = QW_Q
$$

Shape: `(batch, seq_q, d_model)`

$$
K_{\text{proj}} = KW_K
$$

Shape: `(batch, seq_k, d_model)`

$$
V_{\text{proj}} = VW_V
$$

Shape: `(batch, seq_k, d_model)`

### 2. Split into Heads

$$
Q_{\text{heads}}
$$

Shape: `(batch, num_heads, seq_q, d_k)`

$$
K_{\text{heads}}
$$

Shape: `(batch, num_heads, seq_k, d_k)`

$$
V_{\text{heads}}
$$

Shape: `(batch, num_heads, seq_k, d_v)`

### 3. Scaled Dot-Product Attention (Per Head, in Parallel)

Output shape:

`(batch, num_heads, seq_q, d_v)`

Weights shape:

`(batch, num_heads, seq_q, seq_k)`

### 4. Concatenate Heads

$$
\text{concat}
$$

Shape:

`(batch, seq_q, num_heads * d_v)`

$$
=
(batch, seq_q, d_{\text{model}})
$$

### 5. Final Linear Projection

Output shape:

`(batch, seq_q, d_model)`

---

## Transformer Base Parameters

- $d_{\text{model}} = 512$
- $num\_heads = 8$
- $d_k = d_v = 512 / 8 = 64$

In [5]:
print("Transformer Base — Multi-Head Attention dimensions:")
print(f"d_model= 512")
print(f"num_heads= 8")
print(f"d_k = d_v= 512 / 8 = 64")
print(f"W_Q shape= (512, 512) [= d_model × d_model]")
print(f"W_K shape= (512, 512)")
print(f"W_V shape= (512, 512)")
print(f"W_O shape= (512, 512)")
print(f"Parameters/layer = 4 × 512 × 512 = {4 * 512 * 512:,}")

Transformer Base — Multi-Head Attention dimensions:
d_model= 512
num_heads= 8
d_k = d_v= 512 / 8 = 64
W_Q shape= (512, 512) [= d_model × d_model]
W_K shape= (512, 512)
W_V shape= (512, 512)
W_O shape= (512, 512)
Parameters/layer = 4 × 512 × 512 = 1,048,576


# 3: MultiHeadAttention Class

Implements multi-head attention as a `tf.keras.layers.Layer`  
subclass so it integrates cleanly with Keras model building,  
weight saving, and training.

> **NOTE:** I did NOT use `tf.keras.layers.MultiHeadAttention`.  
> The projection and split logic is implemented manually.

In [6]:
class MultiHeadAttention(tf.keras.layers.Layer):
    """
    Multi-Head Attention from scratch.

    MultiHead(Q, K, V) = Concat(head_1, ..., head_h) @ W_O
    where head_i = Attention(Q @ W_Q_i, K @ W_K_i, V @ W_V_i)

    Args:
        d_model  : model dimension (e.g. 512)
        num_heads: number of attention heads (e.g. 8)

    Input shapes:
        q: (batch, seq_q, d_model)
        k: (batch, seq_k, d_model)
        v: (batch, seq_k, d_model)
        mask: broadcastable to (batch, num_heads, seq_q, seq_k)

    Output shape:
        (batch, seq_q, d_model)
    """

    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0, (
            f"d_model ({d_model}) must be divisible by "
            f"num_heads ({num_heads})"
        )

        self.num_heads = num_heads
        self.d_model = d_model

        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads

        # Four learned projection matrices
        # Each is a Dense layer with NO bias and NO activation.
        # W_Q, W_K, W_V: project input to (batch, seq, d_model)
        #                 then we split into heads.
        # W_O: project concatenated heads back to d_model.
        self.W_Q = tf.keras.layers.Dense(d_model, use_bias=False, name='W_Q')
        self.W_K = tf.keras.layers.Dense(d_model, use_bias=False, name='W_K')
        self.W_V = tf.keras.layers.Dense(d_model, use_bias=False, name='W_V')
        self.W_O = tf.keras.layers.Dense(d_model, use_bias=False, name='W_O')

    def split_heads(self, x, batch_size):
        """
        Split the last dimension into (num_heads, d_k).

        Reshape from (batch, seq, d_model)
                  to (batch, num_heads, seq, d_k)

        Args:
            x: (batch, seq, d_model)
            batch_size: integer batch size

        Returns:
            (batch, num_heads, seq, d_k)
        """
        seq_len = tf.shape(x)[1]
        x = tf.reshape(x, (batch_size, seq_len, self.num_heads, self.d_k))

        x = tf.transpose(x, perm=[0, 2, 1, 3])
        return x

    def call(self,q,k,v,mask=None,training = False,):
        """
        Forward pass of Multi-Head Attention.

        Args:
            q: query  tensor (batch, seq_q, d_model)
            k: key    tensor (batch, seq_k, d_model)
            v: value  tensor (batch, seq_k, d_model)
            mask: optional attention mask
            training: passed to sub-layers (not used here directly
                      but needed for Dropout in encoder/decoder)

        Returns:
            output          : (batch, seq_q, d_model)
            attention_weights: (batch, num_heads, seq_q, seq_k)
        """

        batch_size = tf.shape(q)[0]

        # 1. Linear Projections
        # Qproj=QW_Q
        # Shape: (batch, seq_q, d_model)

        # Kproj=KW_K
        # Shape: (batch, seq_k, d_model)

        # Vproj=VW_V
        # Shape: (batch, seq_k, d_model)
        Q_proj = self.W_Q(q)
        K_proj = self.W_K(k)
        V_proj = self.W_V(v)

        # 2. Split into heads:
        Q_heads = self.split_heads(Q_proj, batch_size)
        # Q_heads shape: (batch, num_heads, seq_q, d_k)
        K_heads = self.split_heads(K_proj, batch_size)
        # K_heads shape: (batch, num_heads, seq_k, d_k)
        V_heads = self.split_heads(V_proj, batch_size)
        # V_heads shape: (batch, num_heads, seq_k, d_v)

        # 3. Scaled Dot-Product Attention (Per Head, in Parallel)
        # Output shape:
        # (batch, num_heads, seq_q, d_v)

        # Weights shape:
        # (batch, num_heads, seq_q, seq_k)
        attention_op, attention_wts = scaled_dot_product_attention(Q_heads, K_heads, V_heads, mask=mask)

        # 4. Concatenate Heads
        #     # Shape:
        #     # (batch, seq_q, num_heads * d_v)=(batch,seqq,dmodel)

        attention_op = tf.transpose(attention_op, perm=[0, 2, 1, 3])
        # attention_op shape: (batch, seq_q, num_heads, d_v)

        seq_q = tf.shape(attention_op)[1]

        attention_op = tf.reshape(attention_op, (batch_size, seq_q, self.d_model))
        # attention_op shape: (batch, seq_q, d_model

        # 5. Final linear projection
        output = self.W_O(attention_op)
        # output shape: (batch, seq_q, d_model)

        return output, attention_wts

    def get_config(self):
        """Required for model saving/loading."""

        config = super().get_config()
        config.update({
            'd_model' : self.d_model,
            'num_heads': self.num_heads,
        })

        return config

# 4: Shape Tests

Verifies every tensor shape through the full forward pass  
using Transformer Base dimensions

In [7]:
D_MODEL   = 512
NUM_HEADS = 8
D_K       = D_MODEL // NUM_HEADS   # 64
BATCH     = 2
SEQ_Q     = 10
SEQ_K     = 12

In [8]:
D_K

64

In [9]:
assert D_MODEL // NUM_HEADS == 64, \
    f"d_k should be 64, got {D_MODEL // NUM_HEADS}"
print(f"\nConfiguration:")
print(f"  d_model   = {D_MODEL}")
print(f"  num_heads = {NUM_HEADS}")
print(f"  d_k = d_v = {D_K}   (= {D_MODEL} / {NUM_HEADS})")
print(f"  batch     = {BATCH}")
print(f"  seq_q     = {SEQ_Q}")
print(f"  seq_k     = {SEQ_K}")


Configuration:
  d_model   = 512
  num_heads = 8
  d_k = d_v = 64   (= 512 / 8)
  batch     = 2
  seq_q     = 10
  seq_k     = 12


In [10]:
mha = MultiHeadAttention(d_model=D_MODEL, num_heads=NUM_HEADS)

In [11]:
tf.random.set_seed(42)
Q = tf.random.normal((BATCH, SEQ_Q, D_MODEL))
K = tf.random.normal((BATCH, SEQ_K, D_MODEL))
V = tf.random.normal((BATCH, SEQ_K, D_MODEL))

## Test 1: No mask  

In [12]:
output, weights = mha(Q, K, V, mask=None)

In [13]:
assert output.shape == (BATCH, SEQ_Q, D_MODEL), \
    f"Output shape wrong: {output.shape}"
assert weights.shape == (BATCH, NUM_HEADS, SEQ_Q, SEQ_K), \
    f"Weights shape wrong: {weights.shape}"

print(f"\nTest 1 — No mask:")
print(f"output shape : {output.shape}   <CHECK>")
print(f"weights shape : {weights.shape}  <CHECK>")


Test 1 — No mask:
output shape : (2, 10, 512)   <CHECK>
weights shape : (2, 8, 10, 12)  <CHECK>


## Test 2: Self-attention (seq_q == seq_k)

In [14]:
SEQ = 15
X = tf.random.normal((BATCH, SEQ, D_MODEL))
out_self, w_self = mha(X, X, X, mask=None)

In [15]:
assert out_self.shape == (BATCH, SEQ, D_MODEL)
assert w_self.shape == (BATCH, NUM_HEADS, SEQ, SEQ)

In [16]:
print(f"\nTest 2 — Self-attention (seq={SEQ}):")
print(f"output  shape : {out_self.shape}   <W>")
print(f"weights shape : {w_self.shape}  <W>")


Test 2 — Self-attention (seq=15):
output  shape : (2, 15, 512)   <W>
weights shape : (2, 8, 15, 15)  <W>


##Test 3: With padding mask

In [17]:
seq_ids  = tf.constant(
    [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 0, 0],
     [1, 2, 3, 0, 0, 0, 0, 0, 0,  0, 0, 0]],
    dtype=tf.int32
)

In [18]:
pad_mask = create_padding_mask(seq_ids)

In [19]:
out_m, w_m = mha(Q, K, V, mask=pad_mask)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'multi_head_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [20]:
assert out_m.shape == (BATCH, SEQ_Q, D_MODEL)
assert w_m.shape   == (BATCH, NUM_HEADS, SEQ_Q, SEQ_K)

print(f"\nTest 3 — With padding mask:")
print(f"output  shape : {out_m.shape}   ")
print(f"weights shape : {w_m.shape} ")


Test 3 — With padding mask:
output  shape : (2, 10, 512)   
weights shape : (2, 8, 10, 12) 


##Test 4: With causal mask

In [21]:
SEQ_CAUSAL = 10

In [22]:
X_c   = tf.random.normal((BATCH, SEQ_CAUSAL, D_MODEL))

In [23]:
c_mask = create_look_ahead_mask(SEQ_CAUSAL)

In [24]:
out_c, w_c = mha(X_c, X_c, X_c, mask=c_mask)

In [25]:
assert out_c.shape == (BATCH, SEQ_CAUSAL, D_MODEL)
assert w_c.shape   == (BATCH, NUM_HEADS, SEQ_CAUSAL, SEQ_CAUSAL)

print(f"\nTest 4 — Causal mask (decoder self-attention):")
print(f"output  shape : {out_c.shape} ")
print(f"weights shape : {w_c.shape} ")


Test 4 — Causal mask (decoder self-attention):
output  shape : (2, 10, 512) 
weights shape : (2, 8, 10, 10) 


#   5: Explicit $d_k$ Verification

The paper states:

$$
d_k = d_v = \frac{d_{\text{model}}}{h}
= \frac{512}{8}
= 64
$$

This cell makes that verification explicit and permanent.

In [26]:
configs = [
    ("Transformer Base", 512, 8),
    ("Transformer Big", 1024, 16),
    ("Colab Small",128, 4),
    ("Colab Medium",256, 8),
]

In [27]:
print(f"\n{'Model':<25} {'d_model':>8} {'heads':>6} {'d_k':>6} {'Check'}")

for name, d_model, heads in configs:
    d_k    = d_model // heads
    check  = "<W>" if d_model % heads == 0 else "X NOT DIVISIBLE"
    print(f"{name:<25} {d_model:>8} {heads:>6} {d_k:>6} {check}")

print()


Model                      d_model  heads    d_k Check
Transformer Base               512      8     64 <W>
Transformer Big               1024     16     64 <W>
Colab Small                    128      4     32 <W>
Colab Medium                   256      8     32 <W>



In [28]:
mha_check = MultiHeadAttention(d_model=512, num_heads=8)
_ = mha_check(
    tf.zeros((1, 1, 512)),
    tf.zeros((1, 1, 512)),
    tf.zeros((1, 1, 512))
)

In [29]:
print(f"d_k = {mha_check.d_k} (512 / 8 = 64) <W>")
print(f"  d_v = {mha_check.d_v}(512 / 8 = 64) <W>")

d_k = 64 (512 / 8 = 64) <W>
  d_v = 64(512 / 8 = 64) <W>


In [30]:
print("Learned weight shapes:")
for w in mha_check.trainable_weights:
    print(f"  {w.name:<30} shape={w.shape}  "
          f"params={w.shape[0]*w.shape[1]:,}")

Learned weight shapes:
  kernel                         shape=(512, 512)  params=262,144
  kernel                         shape=(512, 512)  params=262,144
  kernel                         shape=(512, 512)  params=262,144
  kernel                         shape=(512, 512)  params=262,144


In [31]:
total = sum(
    tf.size(w).numpy() for w in mha_check.trainable_weights
)

print(f"\nTotal parameters (one MHA layer): {total:,}")

print(f"Expected: 4 × 512 × 512 = {4 * 512 * 512:,}")

assert total == 4 * 512 * 512, \
    f"Parameter count wrong: {total} vs {4 * 512 * 512}"
print("Parameter count verified.")


Total parameters (one MHA layer): 1,048,576
Expected: 4 × 512 × 512 = 1,048,576
Parameter count verified.


#   6: Numerical Behavior Tests

Verifies multi-head attention properties:

1. Each head produces different attention patterns
2. Attention weights per head sum to 1.0 across key dim
3. Masked positions receive ~0 weight across all heads
4. Output is not just a copy of input (transformation applied)

In [32]:
tf.random.set_seed(0)
BATCH, SEQ, D_MODEL, NUM_HEADS = 2, 8, 512, 8

In [33]:
mha_test = MultiHeadAttention(d_model=D_MODEL, num_heads=NUM_HEADS)
X = tf.random.normal((BATCH, SEQ, D_MODEL))

output, weights = mha_test(X, X, X)

##Test 1: Weights sum to 1.0 per head

In [34]:
weight_sums = tf.reduce_sum(weights, axis=-1)
# shape: (batch, num_heads, seq)
max_dev = tf.reduce_max(tf.abs(weight_sums - 1.0)).numpy()

print(f"\nTest 1 — Attention weights sum to 1.0 per head:")
print(f"Max deviation from 1.0 : {max_dev:.2e}")
assert max_dev < 1e-5
print(f"Passed")


Test 1 — Attention weights sum to 1.0 per head:
Max deviation from 1.0 : 1.79e-07
Passed


In [35]:
weight_sums

<tf.Tensor: shape=(2, 8, 8), dtype=float32, numpy=
array([[[0.99999994, 0.9999999 , 1.        , 0.9999999 , 0.9999999 ,
         0.99999994, 1.        , 0.9999998 ],
        [1.        , 1.        , 0.9999999 , 1.        , 1.0000001 ,
         1.        , 1.0000001 , 1.        ],
        [0.9999999 , 1.        , 0.99999994, 1.        , 0.9999999 ,
         1.        , 1.        , 1.        ],
        [1.        , 0.9999999 , 1.0000001 , 1.        , 1.0000001 ,
         1.        , 1.        , 0.9999999 ],
        [0.99999994, 1.0000001 , 0.9999999 , 0.99999994, 0.9999999 ,
         0.9999999 , 1.        , 0.9999998 ],
        [1.        , 1.        , 0.9999999 , 1.        , 0.99999994,
         0.99999994, 1.        , 1.        ],
        [0.9999998 , 1.        , 1.        , 0.9999999 , 1.        ,
         1.        , 0.9999999 , 1.0000001 ],
        [1.        , 1.        , 1.        , 1.        , 1.        ,
         1.        , 1.        , 0.99999994]],

       [[1.        , 0.9999

##Test 2: Different heads produce different patterns

In [36]:
head_0_weights = weights[0, 0, :, :].numpy()   # head 0
head_1_weights = weights[0, 1, :, :].numpy()   # head 1
head_7_weights = weights[0, 7, :, :].numpy()   # head 7

In [37]:
diff_01 = np.abs(head_0_weights - head_1_weights).mean()
diff_07 = np.abs(head_0_weights - head_7_weights).mean()

In [38]:
print(f"Mean abs diff (head 0 vs head 1) : {diff_01:.6f}")
print(f"Mean abs diff (head 0 vs head 7) : {diff_07:.6f}")

assert diff_01 > 1e-6 and diff_07 > 1e-6, \
    "Heads are producing identical patterns — weights not initialised differently"

print(f"Passed — heads produce distinct attentio patterns")


Mean abs diff (head 0 vs head 1) : 0.121083
Mean abs diff (head 0 vs head 7) : 0.113813
Passed — heads produce distinct attentio patterns


##Test 3: Masked positions receive ~0 across ALL heads

In [39]:
seq_ids  = tf.constant(
    [[1, 2, 3, 4, 5, 0, 0, 0],
     [1, 2, 0, 0, 0, 0, 0, 0]],
    dtype=tf.int32
)

In [40]:
pad_mask = create_padding_mask(seq_ids)
# shape: (2, 1, 1, 8)

In [41]:
X_pad = tf.random.normal((2, SEQ, D_MODEL))
_, w_pad = mha_test(X_pad, X_pad, X_pad, mask=pad_mask)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'multi_head_attention_2' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [42]:
max_pad_weight_b0 = w_pad[0, :, :, 5:].numpy().max()

In [43]:
print(f"Max weight at PAD positions (batch 0) : {max_pad_weight_b0:.2e}")
assert max_pad_weight_b0 < 1e-7
print(f"Passed")

Max weight at PAD positions (batch 0) : 0.00e+00
Passed


In [44]:
output_np = output.numpy()
input_np = X.numpy()
mean_diff = np.abs(output_np - input_np).mean()

print(f"\nTest 4 — Output is a transformation of input (not identity):")
print(f"Mean abs diff (output vs input) : {mean_diff:.6f}")
assert mean_diff > 1e-3, "Output is too similar to input — projection may be identity"
print(f"Passed")



Test 4 — Output is a transformation of input (not identity):
Mean abs diff (output vs input) : 0.878862
Passed


# 7: Causal Mask Test

Verifies that in decoder self-attention mode, no head  
attends to any future position across the entire batch.

In [45]:
tf.random.set_seed(3)
BATCH_C, SEQ_C, D_MODEL_C, NUM_HEADS_C = 2, 8, 512, 8

mha_causal = MultiHeadAttention(d_model=D_MODEL_C, num_heads=NUM_HEADS_C)
X_c= tf.random.normal((BATCH_C, SEQ_C, D_MODEL_C))
causal_mask = create_look_ahead_mask(SEQ_C)

_, w_causal = mha_causal(X_c, X_c, X_c, mask=causal_mask)

w_np= w_causal.numpy()
violations = 0

for b in range(BATCH_C):
    for h in range(NUM_HEADS_C):
        for i in range(SEQ_C):
            for j in range(i + 1, SEQ_C):
                if w_np[b, h, i, j] > 1e-7:
                    violations += 1
                    print(f"X Violation: batch={b} head={h} "
                          f"query={i} attends to future key={j} "
                          f"weight={w_np[b,h,i,j]:.6f}")

if violations == 0:
    print(f"  Checked {BATCH_C} batches x "
          f"{NUM_HEADS_C} heads x "
          f"{SEQ_C} query positions")
    print(f" No causal violations across any head.")
else:
    print(f"\n  X {violations} causal violations found.")


for h in range(NUM_HEADS_C):
    for i in range(SEQ_C):
        assert w_np[0, h, i, i] > 0, \
            f"Head {h}, position {i} has zero self-attention weight"

print(f" All positions attend to themselves across all heads.")

  Checked 2 batches x 8 heads x 8 query positions
 No causal violations across any head.
 All positions attend to themselves across all heads.


/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'multi_head_attention_3' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


# 8: Gradient Flow Test

Verifies gradients flow through the full `MultiHeadAttention`  
layer back to the input and through all 4 weight matrices.

In [46]:
tf.random.set_seed(42)
BATCH_G, SEQ_G, D_G, H_G = 2, 6, 512, 8

mha_grad = MultiHeadAttention(d_model=D_G, num_heads=H_G)

Q_g = tf.random.normal((BATCH_G, SEQ_G, D_G))
K_g = tf.random.normal((BATCH_G, SEQ_G, D_G))
V_g = tf.random.normal((BATCH_G, SEQ_G, D_G))

with tf.GradientTape() as tape:
    tape.watch(Q_g)
    tape.watch(K_g)
    tape.watch(V_g)
    output_g, _ = mha_grad(Q_g, K_g, V_g, training=True)
    loss_g      = tf.reduce_sum(output_g)


input_grads = tape.gradient(loss_g, [Q_g, K_g, V_g])

print("\nGradients w.r.t. inputs:")
all_ok = True
for name, grad in zip(['Q', 'K', 'V'], input_grads):
    if grad is None:
        print(f"  {name}: X None")
        all_ok = False
    else:
        norm    = tf.norm(grad).numpy()
        has_nan = tf.reduce_any(tf.math.is_nan(grad)).numpy()
        status  = "W" if (norm > 0 and not has_nan) else "X"
        print(f"  {name}: norm={norm:.4f}  nan={has_nan}  {status}")
        if norm == 0 or has_nan:
            all_ok = False

with tf.GradientTape() as tape2:
    out2, _ = mha_grad(Q_g, K_g, V_g, training=True)
    loss2 = tf.reduce_sum(out2)

weight_grads = tape2.gradient(loss2, mha_grad.trainable_variables)

print("\nGradients w.r.t. learned weights:")
for w, g in zip(mha_grad.trainable_variables, weight_grads):
    if g is None:
        print(f"  {w.name}: X None")
        all_ok = False
    else:
        norm = tf.norm(g).numpy()
        has_nan = tf.reduce_any(tf.math.is_nan(g)).numpy()
        status = "W" if (norm > 0 and not has_nan) else "X"
        print(f"  {w.name:<35} norm={norm:.4f}  {status}")
        if norm == 0 or has_nan:
            all_ok = False

if all_ok:
    print(f"\nW Gradient flow test passed.")
else:
    print(f"\nX Gradient flow test FAILED.")


Gradients w.r.t. inputs:
  Q: norm=34.8721  nan=False  W
  K: norm=35.2673  nan=False  W
  V: norm=84.3205  nan=False  W

Gradients w.r.t. learned weights:
  kernel                              norm=768.2764  W
  kernel                              norm=788.3366  W
  kernel                              norm=1954.9429  W
  kernel                              norm=1801.5785  W

W Gradient flow test passed.


# 9: Three Attention Modes

The Transformer uses `MultiHeadAttention` in THREE distinct  
modes. This cell verifies all three explicitly.

## Mode 1 — ENCODER SELF-ATTENTION

- Q = K = V = encoder input
- Mask: padding mask on encoder input
- Each position attends to all non-PAD positions.

## Mode 2 — DECODER MASKED SELF-ATTENTION

- Q = K = V = decoder input (so far)
- Mask: combined look-ahead + padding mask
- Each position attends only to itself and past positions.

## Mode 3 — DECODER CROSS-ATTENTION (ENCODER-DECODER)

- Q = decoder state
- K = V = encoder output
- Mask: padding mask on ENCODER input (not decoder)
- Decoder queries attend to all non-PAD encoder positions.

In [47]:
tf.random.set_seed(7)

In [48]:
D_MODEL   = 512
NUM_HEADS = 8
BATCH     = 2
SRC_LEN   = 12   # encoder / source sequence length
TGT_LEN   = 8    # decoder / target sequence length


In [49]:
mha_modes = MultiHeadAttention(d_model=D_MODEL, num_heads=NUM_HEADS)

In [50]:
encoder_output = tf.random.normal((BATCH, SRC_LEN, D_MODEL))
decoder_state = tf.random.normal((BATCH, TGT_LEN, D_MODEL))

In [51]:
src_ids = tf.constant(
    [[1,2,3,4,5,6,7,8,9,10,0,0],
     [1,2,3,4,5,6,0,0,0, 0,0,0]],
    dtype=tf.int32
)

In [52]:
tgt_ids = tf.constant(
    [[1,2,3,4,5,0,0,0],
     [1,2,3,0,0,0,0,0]],
    dtype=tf.int32
)

##Mode 1: Encoder Self-Attention

In [53]:
enc_pad_mask= create_padding_mask(src_ids)
out_enc, w_enc = mha_modes(
    encoder_output, encoder_output, encoder_output,
    mask=enc_pad_mask
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'multi_head_attention_5' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [54]:
assert out_enc.shape == (BATCH, SRC_LEN, D_MODEL)
assert w_enc.shape   == (BATCH, NUM_HEADS, SRC_LEN, SRC_LEN)
print(f"\nMode 1 — Encoder Self-Attention:")
print(f"Q=K=V=encoder_output  mask=padding")
print(f"output : {out_enc.shape}   DONE")
print(f"weights : {w_enc.shape}  DON")


Mode 1 — Encoder Self-Attention:
Q=K=V=encoder_output  mask=padding
output : (2, 12, 512)   DONE
weights : (2, 8, 12, 12)  DON


##Mode 2: Decoder Masked Self-Attention

In [55]:
dec_mask= create_decoder_mask(tgt_ids, tgt_len=TGT_LEN)
out_dec_self, w_ds   = mha_modes(
    decoder_state, decoder_state, decoder_state,
    mask=dec_mask
)
assert out_dec_self.shape == (BATCH, TGT_LEN, D_MODEL)
assert w_ds.shape== (BATCH, NUM_HEADS, TGT_LEN, TGT_LEN)
print(f"\nMode 2 — Decoder Masked Self-Attention:")
print(f"Q=K=V=decoder_state   mask=causal+padding")
print(f"output: {out_dec_self.shape}   ✅")
print(f"weights: {w_ds.shape}  ✅")


Mode 2 — Decoder Masked Self-Attention:
Q=K=V=decoder_state   mask=causal+padding
output: (2, 8, 512)   ✅
weights: (2, 8, 8, 8)  ✅


##Mode 3: Decoder Cross-Attention

In [56]:
cross_mask= create_padding_mask(src_ids)

out_cross, w_cross= mha_modes(
    decoder_state, encoder_output, encoder_output,
    mask=cross_mask
)

assert out_cross.shape== (BATCH, TGT_LEN, D_MODEL)
assert w_cross.shape== (BATCH, NUM_HEADS, TGT_LEN, SRC_LEN)

print(f"\nMode 3 — Decoder Cross-Attention:")
print(f"Q=decoder_state  K=V=encoder_output  mask=src_padding")
print(f"output: {out_cross.shape} DONE")
print(f"weights: {w_cross.shape}  DONE")
print(f"attending to encoder key positions.")


Mode 3 — Decoder Cross-Attention:
Q=decoder_state  K=V=encoder_output  mask=src_padding
output: (2, 8, 512) DONE
weights: (2, 8, 8, 12)  DONE
attending to encoder key positions.


# 10 — Save to Drive

In [57]:
MHA_PY = os.path.join(PROJECT_ROOT, 'multi_head_attention.py')

In [58]:
code = '''"""
multi_head_attention.py — Multi-Head Attention
Transformer Reproduction Project — Phase 5

Usage:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from multi_head_attention import MultiHeadAttention
"""

import tensorflow as tf
from attention import scaled_dot_product_attention


class MultiHeadAttention(tf.keras.layers.Layer):
    """
    Multi-Head Attention from scratch.
    MultiHead(Q,K,V) = Concat(head_1,...,head_h) @ W_O
    where head_i = Attention(Q @ W_Q_i, K @ W_K_i, V @ W_V_i)

    Args:
        d_model: model dimension (e.g. 512)
        num_heads: number of attention heads (e.g. 8)
    """

    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        self.W_Q = tf.keras.layers.Dense(d_model, use_bias=False, name="W_Q")
        self.W_K = tf.keras.layers.Dense(d_model, use_bias=False, name="W_K")
        self.W_V = tf.keras.layers.Dense(d_model, use_bias=False, name="W_V")
        self.W_O = tf.keras.layers.Dense(d_model, use_bias=False, name="W_O")

    def split_heads(self, x, batch_size):
        seq_len = tf.shape(x)[1]
        x = tf.reshape(x, (batch_size, seq_len, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask=None, training=False):
        batch_size = tf.shape(q)[0]
        Q_proj = self.W_Q(q)
        K_proj = self.W_K(k)
        V_proj = self.W_V(v)
        Q_heads = self.split_heads(Q_proj, batch_size)
        K_heads = self.split_heads(K_proj, batch_size)
        V_heads = self.split_heads(V_proj, batch_size)
        attn_out, attn_weights = scaled_dot_product_attention(
            Q_heads, K_heads, V_heads, mask=mask
        )
        attn_out = tf.transpose(attn_out, perm=[0, 2, 1, 3])
        seq_q = tf.shape(attn_out)[1]
        attn_out = tf.reshape(
            attn_out, (batch_size, seq_q, self.num_heads * self.d_v)
        )
        output = self.W_O(attn_out)
        return output, attn_weights

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model, "num_heads": self.num_heads})
        return config
'''


In [59]:
with open(MHA_PY, 'w', encoding='utf-8') as f:
    f.write(code)

In [61]:
import importlib
import multi_head_attention
importlib.reload(multi_head_attention)
from multi_head_attention import MultiHeadAttention as MHA_loaded

mha_reload = MHA_loaded(d_model=512, num_heads=8)
tf.random.set_seed(0)
x_test     = tf.random.normal((2, 10, 512))
out_r, w_r = mha_reload(x_test, x_test, x_test)

assert out_r.shape == (2, 10, 512)
assert w_r.shape   == (2, 8, 10, 10)

print(f"✅ multi_head_attention.py saved and reloaded.")
print(f"   Path: {MHA_PY}")
print(f"\nFinal shape verification (Transformer Base):")
print(f"  Input  : (2, 10, 512)")
print(f"  Output : {out_r.shape}   ✅")
print(f"  Weights: {w_r.shape}  ✅")

✅ multi_head_attention.py saved and reloaded.
   Path: /content/drive/MyDrive/Transformer_Mine/multi_head_attention.py

Final shape verification (Transformer Base):
  Input  : (2, 10, 512)
  Output : (2, 10, 512)   ✅
  Weights: (2, 8, 10, 10)  ✅
